# Balanceamento de eventos extremos

Este notebook explica como o sampler balanceado altera a frequencia de janelas de chuva durante o treinamento. Ele nao substitui a avaliacao no teste cronologico, que deve manter a distribuicao natural.

## Classes de intensidade

Cada janela de treino e classificada pelo maior `m15` observado nos cinco horizontes futuros:

| Classe | Intervalo em mm/15 min |
|---|---|
| Fraca/sem chuva | menor que 1,25 |
| Moderada | 1,25 a menor que 6,25 |
| Forte | 6,25 a menor que 12,5 |
| Extrema | maior ou igual a 12,5 |

Os numeros abaixo sao ilustrativos e inspirados na auditoria historica 2012-2021. Recalcule-os quando o target, o crop ou os anos mudarem.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

class_counts = pd.Series({
    'fraca/sem chuva': 54075,
    'moderada': 4375,
    'forte': 934,
    'extrema': 513,
}, name='janelas')

distribution = class_counts.to_frame()
distribution['proporcao_natural'] = distribution['janelas'] / distribution['janelas'].sum()
display(distribution)

ax = distribution['janelas'].plot.bar(figsize=(8, 4), color='#d97706')
ax.set(xlabel='Classe', ylabel='Janelas', title='Distribuicao natural das janelas de treino')
plt.xticks(rotation=0); plt.tight_layout()

## O que faz o sampler balanceado

O `WeightedRandomSampler` usa pesos inversamente proporcionais ao numero de janelas em cada classe e sorteia com reposicao. Em expectativa, cada classe ocupa 25% dos draws de uma epoca. Isso nao cria novos eventos: uma mesma janela extrema pode ser sorteada varias vezes.

In [ ]:
total_draws = int(class_counts.sum())
balanced_draws = pd.Series(total_draws / len(class_counts), index=class_counts.index, name='draws_esperados')
comparison = distribution.join(balanced_draws)
comparison['proporcao_balanceada'] = comparison['draws_esperados'] / total_draws
comparison['repeticoes_esperadas_por_janela'] = comparison['draws_esperados'] / comparison['janelas']
display(comparison)

ax = comparison[['proporcao_natural', 'proporcao_balanceada']].plot.bar(figsize=(9, 4))
ax.set(xlabel='Classe', ylabel='Proporcao por epoca', title='Distribuicao natural versus sampler balanceado')
plt.xticks(rotation=0); plt.tight_layout()

No exemplo, cada janela extrema e sorteada aproximadamente 29 vezes por epoca em expectativa. Isso aumenta a exposicao do modelo a chuva rara, mas tambem eleva o risco de sobreajuste aos poucos eventos unicos disponiveis.

## Sampler e loss ponderada sao mecanismos diferentes

- O **sampler** decide quais janelas entram no batch e com que frequencia.
- A **loss ponderada** aumenta o peso de pixels de maior intensidade dentro de um batch ja sorteado.

Eles podem ser combinados, como em C2, mas essa combinacao deve ser tratada como uma hipotese experimental: ela pode melhorar eventos extremos ou degradar desempenho em chuva fraca.

In [ ]:
# Exemplo de alternativa moderada: metade da distribuicao natural e metade balanceada.
mix = 0.5
mixed_probability = (1 - mix) * comparison['proporcao_natural'] + mix * comparison['proporcao_balanceada']
mixed = pd.DataFrame({
    'natural': comparison['proporcao_natural'],
    'balanceada': comparison['proporcao_balanceada'],
    'mistura_50_50': mixed_probability,
})
display(mixed)

ax = mixed.plot.bar(figsize=(9, 4))
ax.set(xlabel='Classe', ylabel='Probabilidade de sorteio', title='Exemplo de balanceamento moderado')
plt.xticks(rotation=0); plt.tight_layout()

## Como interpretar resultados

1. Registre `class_counts`, limiares e seed no experimento.
2. Avalie sempre no teste sem reamostragem e reporte `n` por intensidade.
3. Compare RMSE, MAE e vies para fraca, moderada, forte e extrema.
4. Verifique se a melhora em extremos nao decorre de grande degradacao nas classes frequentes.
5. Considere probabilidades moderadas quando houver poucas janelas extremas unicas.